# Visium MM — simplified

Same processing and QC steps as `ALL_MM_01July2026_fixed`, with the loading
updated to read the Space Ranger `outs/` on disk instead of the per-sample
`.h5ad` files under `/home/workspace/Xenium/Data`.

Kept exactly as in the original:
- **no spot filtering** — every spot goes through
- **no `log1p`** — it is commented out in the original
- **no HVG selection** — `sc.pp.pca(..., use_highly_variable=False)`
- PCA 30 comps, neighbors k=15 on 30 PCs, then Harmony over `dataset_name`

In [ ]:
import warnings

warnings.filterwarnings("ignore")  # scanpy/anndata/squidpy chatter

import os

import matplotlib.pyplot as plt
import pandas as pd
import scanpy as sc
import squidpy as sq
import harmonypy as hm
import scipy.sparse as sp

## Load

In [ ]:
OUTS_BASE = "/home/workspace/PCD_data/visium/data/outs"
SAMPLES = ["TIS05393", "TIS04788", "TIS05392", "TIS05394", "TIS04789", "TIS05397"]

adatas = {}
for sample in SAMPLES:
    adata = sq.read.visium(os.path.join(OUTS_BASE, sample))
    adata.var_names_make_unique()
    adata.obs_names_make_unique()
    # Space Ranger's library id is not our short sample id; keep it for sc.pl.spatial
    adata.obs["library_id"] = list(adata.uns["spatial"])[0]
    adatas[sample] = adata

combined_adatas = sc.concat(adatas, label="dataset_name", join="outer")
combined_adatas.obs_names_make_unique()

# sc.concat drops uns; restore it so the tissue images stay available
combined_adatas.uns["spatial"] = {lib: meta for adata in adatas.values()
                                  for lib, meta in adata.uns["spatial"].items()}

print(f"{combined_adatas.n_obs} spots x {combined_adatas.n_vars} genes")
print(combined_adatas.obs["dataset_name"].value_counts())

## QC metrics

Annotation only — nothing is filtered, matching the original. The original read
`.h5ad` files that already carried `total_counts` / `n_genes_by_counts`; reading
from `outs/` we compute them here so the UMAP colourings below work.

In [ ]:
combined_adatas.var["mt"] = combined_adatas.var_names.str.startswith("MT-")
sc.pp.calculate_qc_metrics(combined_adatas, qc_vars=["mt"], percent_top=None,
                           log1p=False, inplace=True)

combined_adatas.obs[["n_genes_by_counts", "total_counts", "pct_counts_mt"]].describe()

## Cutoff sweep

Which spots each candidate cutoff would remove, how many, and why — nothing is
filtered here. One column per level, one row per tissue. `CHOSEN` picks the
level the filter cell below applies.

In [ ]:
MIN_GENES  = [10, 20, 30, 40, 50]
MIN_COUNTS = [20, 40, 60, 80, 100]
MAX_MT_PCT = [20, 20, 20, 20, 20]
CHOSEN = 1          # index of the level the filter cell below applies

LEVELS = list(zip(MIN_GENES, MIN_COUNTS, MAX_MT_PCT))
REASONS = ["kept", "low genes", "low counts", "high % mt", "multiple"]
PALETTE = {"kept": "#d9d9d9", "low genes": "#1f77b4", "low counts": "#ff7f0e",
           "high % mt": "#d62728", "multiple": "#9467bd"}

obs = combined_adatas.obs
labels = []
for min_genes, min_counts, max_mt in LEVELS:
    fail = pd.DataFrame({
        "low genes": obs["n_genes_by_counts"] <= min_genes,
        "low counts": obs["total_counts"] <= min_counts,
        "high % mt": obs["pct_counts_mt"] >= max_mt,
    })
    # one label per spot: kept, the single reason it failed, or multiple
    reason = fail.idxmax(axis=1).where(fail.any(axis=1), "kept")
    reason[fail.sum(axis=1) > 1] = "multiple"

    label = f"genes>{min_genes}, counts>{min_counts}, mt<{max_mt}"
    obs[label] = pd.Categorical(reason, categories=REASONS)
    labels.append(label)

print("% of spots cut")
print(obs.groupby("dataset_name", observed=True)[labels]
      .agg(lambda s: 100 * (s != "kept").mean()).round(1).to_string())

fig, axes = plt.subplots(len(SAMPLES), len(LEVELS),
                         figsize=(4.5 * len(LEVELS), 4.5 * len(SAMPLES)), squeeze=False)

for i, sample in enumerate(SAMPLES):
    spots = combined_adatas[combined_adatas.obs["dataset_name"] == sample]
    for j, label in enumerate(labels):
        n_cut = int((spots.obs[label] != "kept").sum())
        sc.pl.spatial(spots, color=label, library_id=spots.obs["library_id"].iloc[0],
                      palette=PALETTE, ax=axes[i, j], show=False,
                      legend_loc="right margin" if (i, j) == (0, len(labels) - 1) else None,
                      title=f"{sample}\n{label}\n{n_cut} cut ({100 * n_cut / spots.n_obs:.0f}%)")

fig.tight_layout()

# what the filter cell below will apply
combined_adatas.obs["cut_reason"] = obs[labels[CHOSEN]]

## Filter spots

Not in the original notebook, which processes every spot. Raise the cutoffs to
drop the near-empty spots; the counts printed below are what survives.

In [ ]:
# combined_adatas = combined_adatas[combined_adatas.obs["cut_reason"] == "kept"].copy()

# print(f"{combined_adatas.n_obs} spots kept")
# combined_adatas.obs["dataset_name"].value_counts()

## Process

In [ ]:
combined_adatas.layers["counts"] = combined_adatas.X.copy()
sc.pp.normalize_total(combined_adatas, inplace=True)
# sc.pp.log1p(combined_adatas)  # left off, as in the original

# normalize_total can densify X
combined_adatas.X = sp.csr_matrix(combined_adatas.X)

sc.pp.pca(combined_adatas, n_comps=30, use_highly_variable=False)
sc.pp.neighbors(combined_adatas, n_neighbors=15, n_pcs=30)
sc.tl.umap(combined_adatas)
sc.tl.leiden(combined_adatas)

In [ ]:
sc.pl.umap(combined_adatas,
           color=["total_counts", "n_genes_by_counts", "leiden", "dataset_name"],
           wspace=0.4)

## Harmony batch correction

In [ ]:
# harmonypy called directly (the scanpy wrapper is a version mismatch)
ho = hm.run_harmony(combined_adatas.obsm["X_pca"].astype("float64"),
                    combined_adatas.obs, "dataset_name")

# Z_corr orientation varies by harmonypy version
z = ho.Z_corr
combined_adatas.obsm["X_pca_harmony"] = z.T if z.shape[0] != combined_adatas.n_obs else z

sc.pp.neighbors(combined_adatas, use_rep="X_pca_harmony")
sc.tl.umap(combined_adatas)
sc.tl.leiden(combined_adatas, flavor="igraph", n_iterations=2, directed=False,
             resolution=0.6)

combined_adatas.obs["leiden"].value_counts().sort_index()

In [ ]:
sc.pl.umap(combined_adatas,
           color=["total_counts", "n_genes_by_counts", "leiden", "dataset_name"],
           wspace=0.15,
           ncols=2
          )

## Marker genes

In [ ]:
sc.tl.rank_genes_groups(combined_adatas, "leiden", method="wilcoxon")
sc.pl.rank_genes_groups(combined_adatas, n_genes=10, sharey=False)

## Cell types

Cluster ids are only meaningful for the run that produced them — check these
against the marker genes above before trusting them.

In [ ]:
# cluster_to_cell_type = {
#     "0": "Erythroblast_1",
#     "1": "Macrophage",
#     "2": "Monocyte",
#     "3": "Endothelial_1",
#     "4": "Monocyte/Macrophage",
#     "5": "Stromal/Fibroblast",
#     "6": "Tumor-Associated Macrophage",
#     "7": "Neoplasm_1",
#     "8": "Erythroblast_2",
#     "9": "Neutrophil",
#     "10": "Neoplasm_2",
#     "11": "Neoplasm_3",
#     "12": "Cytotoxic T Cell",
#     "13": "Neoplasm_4",
#     "14": "Myeloid",
#     "15": "Neoplasm_5",
#     "16": "Endothelial_2",
# }

# combined_adatas.obs["cell_type"] = (
#     combined_adatas.obs["leiden"].map(cluster_to_cell_type).astype("category"))

# sc.pl.umap(combined_adatas, color="cell_type", legend_fontsize=8)

## Spatial

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

for ax, sample in zip(axes.flatten(), SAMPLES):
    spots = combined_adatas[combined_adatas.obs["dataset_name"] == sample]
    sc.pl.spatial(spots, color="leiden", library_id=spots.obs["library_id"].iloc[0],
                  title=sample, ax=ax, show=False)

fig.tight_layout()

In [ ]:
GENES = ["MZB1", "PECAM1", "GNLY"]

for sample in SAMPLES:
    spots = combined_adatas[combined_adatas.obs["dataset_name"] == sample]
    sc.pl.spatial(spots, color=GENES, library_id=spots.obs["library_id"].iloc[0], wspace=0.1)